<a href="https://colab.research.google.com/github/AyeshaThapa/Retail-sales-data/blob/main/FraudDetection2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, average_precision_score
from sklearn.utils import resample
from sklearn.metrics import average_precision_score
import numpy as np

In [ ]:
df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/creditcard.csv")
print(df.head())

   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26       V27       V28 

In [ ]:
#Quick EDA
print("\nColumns:", df.columns.tolist())
print("\nClass distribution:\n", df['Class'].value_counts())
print("\nFraud fraction: {:.6f}".format(df['Class'].mean()))


Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

Class distribution:
 Class
0    284315
1       492
Name: count, dtype: int64

Fraud fraction: 0.001727


In [ ]:
#Preprocessing
X = df.drop(columns=['Class'])
y = df['Class']

In [ ]:
#Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)

In [ ]:
#Scale Time and Amount
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
for col in ['Time', 'Amount']:
    X_train_scaled[col] = scaler.fit_transform(X_train[[col]])
    X_test_scaled[col] = scaler.transform(X_test[[col]])

print("\nTrain / Test sizes:", X_train_scaled.shape, X_test_scaled.shape)


Train / Test sizes: (227845, 30) (56962, 30)


In [ ]:
#Helper for concise evaluation printout
def eval_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    # try to compute ROC-AUC (if possible)
    roc = None
    if hasattr(model, "predict_proba"):
        roc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    elif hasattr(model, "decision_function"):
        roc = roc_auc_score(y_test, model.decision_function(X_test))
    print("\n---", name, "---")
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification report:\n", classification_report(y_test, y_pred, digits=4))
    if roc is not None:
        print("ROC-AUC: {:.6f}".format(roc))
    return roc

In [ ]:
#Supervised model
from sklearn.utils import resample

X_balanced, y_balanced = resample(
    X_train_scaled[y_train==1],
    y_train[y_train==1],
    replace=True,
    n_samples=(y_train==0).sum(),
    random_state=42
)
X_resampled = np.vstack([X_train_scaled[y_train==0].values, X_balanced.values])
y_resampled = np.hstack([y_train[y_train==0].values, y_balanced.values])

mlp = MLPClassifier(hidden_layer_sizes=(50,30), max_iter=200, random_state=42)
mlp.fit(X_resampled, y_resampled)
mlp_roc = eval_model("MLPClassifier (resampled)", mlp, X_test_scaled, y_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(



--- MLPClassifier (resampled) ---
Confusion matrix:
 [[56845    19]
 [   18    80]]
Classification report:
               precision    recall  f1-score   support

           0     0.9997    0.9997    0.9997     56864
           1     0.8081    0.8163    0.8122        98

    accuracy                         0.9994     56962
   macro avg     0.9039    0.9080    0.9059     56962
weighted avg     0.9994    0.9994    0.9994     56962

ROC-AUC: 0.974264


In [ ]:
#Unsupervised anomaly detection baseline
contamination = df['Class'].mean()  # expected fraction of anomalies in data
iso = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
# train on "normal" (non-fraud) transactions only
iso.fit(X_train_scaled[y_train==0])
iso_pred = iso.predict(X_test_scaled)  # -1 anomaly, 1 normal
iso_pred_mapped = (iso_pred == -1).astype(int)  # map anomalies to predicted fraud=1
print("\n--- IsolationForest (unsupervised) ---")
print("Confusion matrix:\n", confusion_matrix(y_test, iso_pred_mapped))
print("Classification report:\n", classification_report(y_test, iso_pred_mapped, digits=4))
# scores (higher -> more anomalous)
iso_scores = -iso.decision_function(X_test_scaled)
print("ROC-AUC (IsolationForest scores): {:.6f}".format(roc_auc_score(y_test, iso_scores)))



--- IsolationForest (unsupervised) ---
Confusion matrix:
 [[56761   103]
 [   65    33]]
Classification report:
               precision    recall  f1-score   support

           0     0.9989    0.9982    0.9985     56864
           1     0.2426    0.3367    0.2821        98

    accuracy                         0.9971     56962
   macro avg     0.6208    0.6675    0.6403     56962
weighted avg     0.9976    0.9971    0.9973     56962

ROC-AUC (IsolationForest scores): 0.953403


In [ ]:
#Logistic Regression with class_weight balanced
lr = LogisticRegression(max_iter=1000, class_weight='balanced', solver='liblinear', random_state=42)
lr.fit(X_train_scaled, y_train)

#Random Forest with class_weight balanced
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

#MLPClassifier trained on resampled data (to balance classes)
X0, y0 = X_train_scaled[y_train==0], y_train[y_train==0]
X1, y1 = X_train_scaled[y_train==1], y_train[y_train==1]
X1_res, y1_res = resample(X1, y1, replace=True, n_samples=len(y0), random_state=42)

X_resampled = np.vstack([X0, X1_res])
y_resampled = np.hstack([y0, y1_res])

mlp_balanced = MLPClassifier(hidden_layer_sizes=(50,30), max_iter=200, random_state=42)
mlp_balanced.fit(X_resampled, y_resampled)

#Precision-Recall summary
print("\n--- Precision-Recall summary (average precision) ---")
for name, model in [("LogReg", lr), ("RandomForest", rf), ("MLP", mlp_balanced)]:
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_test_scaled)[:, 1]
    elif hasattr(model, "decision_function"):
        prob = model.decision_function(X_test_scaled)
    else:
        prob = model.predict(X_test_scaled)
    ap = average_precision_score(y_test, prob)
    print(f"{name} average precision (area under PR): {ap:.4f}")

#If we have IsolationForest scores:
ap_iso = average_precision_score(y_test, iso_scores)
print("IsolationForest average precision:", ap_iso)


--- Precision-Recall summary (average precision) ---
LogReg average precision (area under PR): 0.7189
RandomForest average precision (area under PR): 0.8542
MLP average precision (area under PR): 0.8427
IsolationForest average precision: 0.17412417144901768


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MLPClassifier was fitted without feature names
  warnings.warn(


In [ ]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

lr = LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear", random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("\n--- Logistic Regression ---")
print(classification_report(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))
print("Average Precision:", average_precision_score(y_test, y_prob_lr))

#Decision Tree
dt = DecisionTreeClassifier(class_weight="balanced", max_depth=5, random_state=42)
dt.fit(X_train_scaled, y_train)
y_pred_dt = dt.predict(X_test_scaled)
y_prob_dt = dt.predict_proba(X_test_scaled)[:, 1]

print("\n--- Decision Tree ---")
print(classification_report(y_test, y_pred_dt))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_dt))
print("Average Precision:", average_precision_score(y_test, y_prob_dt))

#Neural Network (MLPClassifier)
mlp = MLPClassifier(hidden_layer_sizes=(50,30), max_iter=200, random_state=42)
mlp.fit(X_train_scaled, y_train)
y_pred_mlp = mlp.predict(X_test_scaled)
y_prob_mlp = mlp.predict_proba(X_test_scaled)[:, 1]

print("\n--- Neural Network (MLPClassifier) ---")
print(classification_report(y_test, y_pred_mlp))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_mlp))
print("Average Precision:", average_precision_score(y_test, y_prob_mlp))


--- Logistic Regression ---
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962

ROC-AUC: 0.9720948047902334
Average Precision: 0.7189348125855011

--- Decision Tree ---
              precision    recall  f1-score   support

           0       1.00      0.97      0.98     56864
           1       0.05      0.88      0.09        98

    accuracy                           0.97     56962
   macro avg       0.52      0.92      0.54     56962
weighted avg       1.00      0.97      0.98     56962

ROC-AUC: 0.9165621985288207
Average Precision: 0.44978383369250374

--- Neural Network (MLPClassifier) ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1

In [ ]:
from joblib import dump
dump(lr, "logreg.joblib")
dump(rf, "rf.joblib")
dump(mlp, "mlp.joblib")
print("Models saved to disk (joblib)")
print("\nDone.")

Models saved to disk (joblib)

Done.
